---

## **Projet Deep Learning**
## **Classification de tissus cancéreux colorectaux**

---


---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans retélécharger si les fichiers sont déjà présents)
- **traçable** (quality gates go/no-go explicites)

---


---

# PARTIE 6 : GRAD-CAM (implémentation maison)

---


---

### Objectif

Ouvrir la "boîte noire" du CNN pour comprendre ce que le modèle regarde quand il prend une décision. Grad-CAM (Gradient-weighted Class Activation Mapping) produit une heatmap qui montre quelles zones de l'image ont le plus contribué à la classification.

### Progression du projet
- **Partie 2 (MLP)** : 63.08% — pixels isolés
- **Partie 3 (CNN)** : 92.38% — le meilleur modèle, exploite les textures
- **Partie 4 (ResNet-18)** : en cours
- **Partie 5 (ViT)** : en cours
- **Partie 6 (Grad-CAM)** : on regarde à l'intérieur du CNN pour comprendre ses décisions

### Consignes du sujet
- Implémentation maison (pas de librairie externe type pytorch-grad-cam)
- Hooks PyTorch pour capturer activations + gradients
- Visualiser heatmaps pour ≥ 3 types de tissus
- Inclure ≥ 1 exemple mal classifié
- **Q6.1** : Décrire visuellement les heatmaps Tumor Epithelium vs Normal Mucosa
- **Q6.2** : Image mal classifiée — Grad-CAM pour classe prédite vs vraie classe, comparer

### Principe de Grad-CAM
1. On fait passer une image dans le CNN
2. On capture les activations de la dernière couche convolutive (ce que le modèle "voit")
3. On calcule les gradients de la classe prédite par rapport à ces activations (ce qui a contribué à la décision)
4. On pondère les activations par les gradients et on obtient une heatmap
5. On superpose la heatmap sur l'image originale pour voir quelles zones comptent

---


In [1]:
# Lien avec le notebook 1 (EDA) — imports
# Version 1.0 — 20 mars 2026

import sys
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import medmnist
from medmnist import PathMNIST
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
print("Imports OK")


Imports OK


In [2]:
# Versions — traçabilité
print(f"Python   : {sys.version.split()[0]}")
print(f"PyTorch  : {torch.__version__}")
print(f"MedMNIST : {medmnist.__version__}")
print(f"NumPy    : {np.__version__}")


Python   : 3.12.10
PyTorch  : 2.10.0+cu128
MedMNIST : 3.0.2
NumPy    : 2.4.3


In [3]:
# Reproductibilité complète (CPU + GPU + cuDNN)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"cuDNN determ. : {torch.backends.cudnn.deterministic if torch.cuda.is_available() else 'N/A'}")


Device        : cuda
cuDNN determ. : True


In [4]:
# Constantes et dataset — calculées dans l'EDA (notebook 1)
DATA_DIR = os.path.join(".", "data")
NORM_MEAN = [0.7405, 0.5330, 0.7058]
NORM_STD  = [0.1237, 0.1768, 0.1244]
N_CLASSES = 9

# Recharger le dataset
train_dataset = PathMNIST(split='train', download=False, root=DATA_DIR)
val_dataset   = PathMNIST(split='val',   download=False, root=DATA_DIR)
test_dataset  = PathMNIST(split='test',  download=False, root=DATA_DIR)

info = train_dataset.info
labels_names = info['label']
CLASS_NAMES = list(labels_names.values())

# Timer global du notebook
notebook_start_time = time.time()

print(f"NORM_MEAN : {NORM_MEAN}")
print(f"NORM_STD  : {NORM_STD}")
print(f"Train : {len(train_dataset)} | Val : {len(val_dataset)} | Test : {len(test_dataset)}")
print("✓ Lien avec notebook 1 établi")


NORM_MEAN : [0.7405, 0.533, 0.7058]
NORM_STD  : [0.1237, 0.1768, 0.1244]
Train : 89996 | Val : 10004 | Test : 7180
✓ Lien avec notebook 1 établi


In [5]:
# Recharger le modèle CNN entraîné (partie 3, sans augmentation)
# On reconstruit l'architecture puis on charge les poids sauvegardés

def create_cnn(n_classes=9):
    model = nn.Sequential(
        # Bloc 1
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.Conv2d(32, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Dropout(0.25),
        # Bloc 2
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Dropout(0.35),
        # Bloc 3
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Dropout(0.5),
        # Classification
        nn.Flatten(),
        nn.Linear(128 * 3 * 3, 128),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128, n_classes)
    )
    return model

cnn_model = create_cnn()
cnn_model.load_state_dict(torch.load(os.path.join(DATA_DIR, 'cnn_model.pth'), map_location=device))
cnn_model = cnn_model.to(device)
cnn_model.eval()

print(f"Modèle CNN chargé depuis {os.path.join(DATA_DIR, 'cnn_model.pth')}")
print(f"Nb paramètres : {sum(p.numel() for p in cnn_model.parameters()):,}")
print("✓ Modèle CNN prêt pour Grad-CAM")


FileNotFoundError: [Errno 2] No such file or directory: '.\\data\\cnn_model.pth'

In [ ]:
# Preprocessing — même que CNN (partie 3)
cnn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

test_cnn = PathMNIST(split='test', transform=cnn_transform, download=False, root=DATA_DIR)
test_loader_cnn = DataLoader(test_cnn, batch_size=1, shuffle=False)  # batch_size=1 pour Grad-CAM

print("✓ Preprocessing Grad-CAM terminé")


In [ ]:
# Implémentation Grad-CAM maison — avec hooks PyTorch
# Pas de librairie externe (consigne du sujet)

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        # Hook pour capturer les activations (forward)
        self.target_layer.register_forward_hook(self._save_activations)
        # Hook pour capturer les gradients (backward)
        self.target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        # Forward pass
        self.model.eval()
        output = self.model(input_tensor)

        # Si pas de classe cible, prendre la classe prédite
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # Backward pass sur la classe cible
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot, retain_graph=True)

        # Pondération des activations par les gradients
        # Global average pooling des gradients
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)

        # ReLU — on ne garde que les contributions positives
        cam = F.relu(cam)

        # Normaliser entre 0 et 1
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam.squeeze().cpu().numpy(), target_class, output

# La dernière couche convolutive de notre CNN est à l'index 21
# (Conv2d 128→128 du bloc 3, avant le MaxPool)
target_layer = cnn_model[21]  # dernière Conv2d du bloc 3

grad_cam = GradCAM(cnn_model, target_layer)
print("✓ Grad-CAM initialisé (target layer : dernière Conv2d du bloc 3)")


In [ ]:
# Fonction de visualisation Grad-CAM
def visualize_gradcam(image_tensor, true_label, grad_cam, class_names, target_class=None):
    """Affiche l'image originale et la heatmap Grad-CAM superposée"""
    # Générer la heatmap
    cam, pred_class, output = grad_cam.generate(image_tensor.unsqueeze(0).to(device), target_class)

    # Dénormaliser l'image pour l'affichage
    img = image_tensor.clone()
    for c in range(3):
        img[c] = img[c] * NORM_STD[c] + NORM_MEAN[c]
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    # Redimensionner la heatmap à la taille de l'image
    import torch.nn.functional as Fnn
    cam_resized = Fnn.interpolate(
        torch.tensor(cam).unsqueeze(0).unsqueeze(0),
        size=(28, 28), mode='bilinear', align_corners=False
    ).squeeze().numpy()

    # Affichage
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Image originale
    axes[0].imshow(img)
    axes[0].set_title(f"Original\nVrai: {class_names[true_label]}")
    axes[0].set_xticks([]); axes[0].set_yticks([])

    # Heatmap seule
    axes[1].imshow(cam_resized, cmap='jet')
    target = target_class if target_class is not None else pred_class
    axes[1].set_title(f"Grad-CAM\nClasse cible: {class_names[target]}")
    axes[1].set_xticks([]); axes[1].set_yticks([])

    # Superposition
    axes[2].imshow(img)
    axes[2].imshow(cam_resized, cmap='jet', alpha=0.5)
    axes[2].set_title(f"Superposition\nPrédit: {class_names[pred_class]}")
    axes[2].set_xticks([]); axes[2].set_yticks([])

    correct = "✓" if pred_class == true_label else "✗"
    plt.suptitle(f"Grad-CAM — {correct} Vrai: {class_names[true_label]} | Prédit: {class_names[pred_class]}", fontsize=13)
    plt.tight_layout()
    plt.show()

print("✓ Fonction visualize_gradcam chargée")


---

## Visualisation Grad-CAM — ≥ 3 types de tissus

---


In [ ]:
# Grad-CAM sur des images bien classifiées — au moins 3 classes différentes
# On cherche des exemples correctement classifiés de classes variées
np.random.seed(SEED)

classes_to_show = [0, 3, 6, 8]  # adipose, lymphocytes, normal mucosa, cancer
# Noms : adipose (blanc), lymphocytes (cellules immunitaires), normal mucosa, cancer

for target_class in classes_to_show:
    # Trouver une image de cette classe dans le test set
    for idx in range(len(test_cnn)):
        img, label = test_cnn[idx]
        true_label = int(label.item()) if hasattr(label, 'item') else int(label[0])
        if true_label == target_class:
            # Vérifier que le modèle la classe correctement
            with torch.no_grad():
                pred = cnn_model(img.unsqueeze(0).to(device)).argmax(1).item()
            if pred == true_label:
                visualize_gradcam(img, true_label, grad_cam, CLASS_NAMES)
                break


---

## Q6.1 — Heatmaps Tumor Epithelium vs Normal Mucosa

---


In [ ]:
# Q6.1 — Comparer les heatmaps Tumor Epithelium (classe 8) vs Normal Mucosa (classe 6)
print("=== Tumor Epithelium ===")
for idx in range(len(test_cnn)):
    img, label = test_cnn[idx]
    true_label = int(label.item()) if hasattr(label, 'item') else int(label[0])
    if true_label == 8:  # cancer
        with torch.no_grad():
            pred = cnn_model(img.unsqueeze(0).to(device)).argmax(1).item()
        if pred == true_label:
            visualize_gradcam(img, true_label, grad_cam, CLASS_NAMES)
            break

print("\n=== Normal Mucosa ===")
for idx in range(len(test_cnn)):
    img, label = test_cnn[idx]
    true_label = int(label.item()) if hasattr(label, 'item') else int(label[0])
    if true_label == 6:  # mucosa
        with torch.no_grad():
            pred = cnn_model(img.unsqueeze(0).to(device)).argmax(1).item()
        if pred == true_label:
            visualize_gradcam(img, true_label, grad_cam, CLASS_NAMES)
            break


---

## Q6.2 — Image mal classifiée : Grad-CAM classe prédite vs vraie classe

---


In [ ]:
# Q6.2 — Trouver une image mal classifiée et comparer les heatmaps
# Grad-CAM pour la classe prédite (pourquoi le modèle s'est trompé)
# vs Grad-CAM pour la vraie classe (qu'aurait-il dû regarder)

for idx in range(len(test_cnn)):
    img, label = test_cnn[idx]
    true_label = int(label.item()) if hasattr(label, 'item') else int(label[0])
    with torch.no_grad():
        pred = cnn_model(img.unsqueeze(0).to(device)).argmax(1).item()
    if pred != true_label:
        print(f"Image mal classifiée trouvée : index {idx}")
        print(f"  Vrai label : {CLASS_NAMES[true_label]} (classe {true_label})")
        print(f"  Prédit     : {CLASS_NAMES[pred]} (classe {pred})")

        # Heatmap pour la classe PRÉDITE (ce que le modèle a vu)
        print(f"\n--- Grad-CAM pour la classe prédite ({CLASS_NAMES[pred]}) ---")
        visualize_gradcam(img, true_label, grad_cam, CLASS_NAMES, target_class=pred)

        # Heatmap pour la VRAIE classe (ce qu'il aurait dû regarder)
        print(f"\n--- Grad-CAM pour la vraie classe ({CLASS_NAMES[true_label]}) ---")
        visualize_gradcam(img, true_label, grad_cam, CLASS_NAMES, target_class=true_label)
        break


---

### Q6.1 — Analyse des heatmaps Tumor Epithelium vs Normal Mucosa

À compléter après exécution — décrire :
- Quelles zones de l'image le modèle regarde pour chaque classe
- Les heatmaps sont-elles concentrées ou diffuses ?
- Le modèle regarde-t-il les bonnes structures (noyaux pour le cancer, glandes pour la mucosa) ?

### Q6.2 — Analyse de l'image mal classifiée

À compléter après exécution — comparer :
- La heatmap de la classe prédite (pourquoi le modèle s'est trompé) vs la vraie classe
- Le modèle regardait-il une zone trompeuse ?
- Est-ce une erreur compréhensible visuellement ?

---


---

## Bilan Partie 6 — Grad-CAM

### Ce qu'on a appris
- À compléter après exécution
- Le modèle regarde-t-il les bonnes zones ?
- Les erreurs sont-elles explicables par les heatmaps ?
- L'interprétabilité est essentielle en contexte clinique — un médecin doit pouvoir vérifier que le modèle prend ses décisions sur des critères pertinents

---


In [ ]:
# Temps total d'exécution du notebook
notebook_total_time = time.time() - notebook_start_time
print(f"Temps total du notebook : {notebook_total_time:.1f}s ({notebook_total_time/60:.1f} min)")
